# 检索增强生成（Retrival-Augmented Generation）

你的LLM知道训练截止前的一切知识，但是不知道你公司的文档、代码库，或者上周的会议纪要。RAG通过将相关的文档召回，然后注入提示词来解决。这是生产级AI最通常被部署的能力。

## 问题描述

对于模型未见过的知识。

微调是一种解决方案，但是开销很大。而且文档一变，模型又要跟着变，微调又要再来一次。你也不知道模型是根据什么做出的回答。

RAG是另外一种解决方案，不动模型。面对问题，先在文档库中搜索相关的文章，然后将它们塞入提示词中，让模型将这些文档作为上下文，并基于此做大。文档库更新只要几分钟，你也能看到哪些文档被召回了。这就是RAG的优势：更便宜、时效性、更容易审计，不依赖模型。

## 基本描述

### RAG模式

整个RAG流程分为四步：
- 查询
- 检索
- 增强提示词
- 生成

每个RAG系统都遵循以上模式，不同的地方在每步的细节：怎么切分文档，怎么做嵌入，怎么查询，以及怎么构造提示词。

### RAG击败微调
|关注点|微调|RAG|
|---|---|---|
|开销|较大$1000-$100000每次训练|很小$0.01~$0.10每查询|
|时效|不重训就过时|几分钟重新索引文档就能更新|
|审计|不知道答案怎么来的|能具体指导引用了哪些文章|
|幻觉|容易产生|答案建立在检索到的文档上|
|数据隐私|训练数据流入权重|关联文档任然在你自己的向量数据库中|

微调永久地改变模型权重；微调临时改变模型上下文。对于大多数任务，临时的上下文更合理。

微调只在仅通过RAG增强提示词做不到的时候使用，比如让模型开始以特定的风格、语调、推理风格生成时。

### 向量相似度
- 余弦相似度
- 点积
- 欧式距离

### 切片策略
- 定长切片
- 语义切片
- 递归切片

切片大小也很重要：
- 太小的切片（64-128）： 每个切片可能缺少上下文。比如“它”指代不明。
- 太大的切片（2048+）： 每个切片包含多个话题，稀释相关性。
- 甜点区（256～512）。

大部分生产级RAG系统使用256～512词元的切片大小，另带有50词元的重叠区。Anthropic 的 RAG指引也推荐这个范围。

### 完整流水线
```mermaid
flowchart TD
subgraph A[文档索引]
    B[文档]--> C[切片] --> D[嵌入] -->E[存储向量+文本]
end
A --相同的嵌入空间--> I[查询]
subgraph F
    G[用户查询]--> H[嵌入]-->I[向量检索（top-k）]-->J[构建增强提示词]--> K[大模型生成]
end
```

### 生产级RAG参数
- 每次查询召回的切片数量在5-10
- 切片大小在256～512词元，其中50个词元重叠
- 上下文预算，2500～5000的词元分配给检索到的内容
- 总提示词：8000～16000词元（系统提示词+检索内容+会话历史+用户输入）
- 嵌入维度，384～3072
- 索引吞吐量，100-1000文档每秒
- 查询延迟，50-200ms来检索，500-3000ms来生成

# 开始编码

## 嵌入示例代码

In [1]:
import math
from collections import Counter

# 衡量text中每个词的重要性，靠词频
def compute_tf(text, vocab):
    words = text.lower().split()
    count = Counter(words)
    total = len(words)
    if total == 0:
        return [0.0] * len(vocab)

    return [count.get(word, 0) / total for word in vocab]

# 衡量词在文档中的重要性，靠逆文档频率
def compute_idf(docs, vocab):
    n = len(docs)
    idf = []
    for word in vocab:
        doc_count = sum(1 for doc in docs if word in doc.lower().split())
        idf.append(math.log((n + 1) / (doc_count + 1)) + 1)
    return idf

def tfidf_embed(text, vocab, idf):
    tf = compute_tf(text, vocab)
    return [t * i for t, i in zip(tf, idf)]

## 检索示例代码

In [ ]:
def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = math.sqrt(sum(x ** 2 for x in a))
    magnitude_b = math.sqrt(sum(y ** 2 for y in b))
    return dot_product / (magnitude_a * magnitude_b)

def search(query_emb, store_embs, top_k=5):
    scores = []
    for i, emb in enumerate(store_embs):
        sim = cosine_similarity(query_emb, emb)
        scores.append((i, sim))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

## 增强提示词示例代码

In [3]:
def build_rag_prompt(query, retrieved_chunks):
    context = "\n\n---\n\n".join(
        f"[Source {i+1}]\n{chunk}"
        for i, chunk in enumerate(retrieved_chunks)
    )
    return (
        "Answer the question based ONLY on the following context.\n"
        "If the context doesn't contain enough information, "
        "say \"I don't have enough information to answer that.\"\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )

## RAG 示例代码

In [ ]:
class RAGPipeline:
    def __init__(self, chunk_size=200, overlap=50, top_k=5):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.top_k = top_k
        self.chunks = []
        self.embeddings = []
        self.vocab = []
        self.idf = []
        self.sources = []

    def index(self, docs, source_names=None):
        all_chunks = []
        all_sources = []

        def chunk_text(text, chunk_size, overlap):
            words = text.lower().split()
            for i in range(0, len(words), chunk_size - overlap):
                chunk = " ".join(words[i:i+chunk_size])
                yield chunk

        for i, doc in enumerate(docs):
            doc_chunks = chunk_text(doc, self.chunk_size, self.overlap)
            for chunk in doc_chunks:
                all_chunks.extend(chunk_text(chunk, self.chunk_size, self.overlap))
                name = source_names[i] if source_names else f"Source {i+1}"
                all_sources.extend([name] * len(doc_chunks))

        self.chunks = all_chunks
        self.sources = all_sources

        self.vocab = list(set(word for chunk in all_chunks for word in chunk.lower().split()))
        self.idf = compute_idf(all_chunks, self.vocab)

        self.embeddings = [
            tfidf_embed(chunk, self.vocab, self.idf)
            for chunk in all_chunks
        ]

        return len(all_chunks)        
    
    
    def query(self, question, top_k=None):
        k = top_k or self.top_k

        query_emb = tfidf_embed(question, self.vocab, self.idf)
        results = search(query_emb, self.embeddings, k)

        retrieved = []
        for idx, score in results:
            retrieved.append({
                "chunk": self.chunks[idx],
                "source": self.sources[idx],
                "score": score,
                "index": idx,
            })

        chunk_texts = [r["chunk"] for r in retrieved]
        prompt = build_rag_prompt(question, chunk_texts)
        # answer = self.llm.invoke(prompt)
        answer = "Fake answer"

        return {
            "question": question,
            "retrieved": retrieved,
            "prompt": prompt,
            "answer": answer,
        }